## Classification: Agent

# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from anthropic import Anthropic
import json
load_dotenv(override=True)

True

In [2]:
claude = Anthropic()

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
# Some lists!

todos = []
completed = []

## Tools

In [4]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result


In [5]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

create_todos_tool = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "input_schema": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [6]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

mark_complete_tool = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "input_schema": {
        'type': 'object',
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'additionalProperties': False
    }
}

In [7]:
def execute_tool_calls(tool_calls):
    tool_results = []
    for tool_call in tool_calls:
        if tool_call.type == "tool_use":
            tool_name = tool_call.name
            arguments = tool_call.input
            tool = globals().get(tool_name)
            result = tool(**arguments) if tool else "Tool not found"
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": result
            })
    return tool_results

In [ ]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

In [ ]:
mark_complete(1, "bought")

In [8]:
tools = [create_todos_tool, mark_complete_tool]

## Prompt (System)

In [9]:
system_prompt = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet? If id is 1
"""
messages = [{"role": "user", "content": user_message}]

In [ ]:
def loop(messages):
    while True:
        response = claude.messages.create(
            model="claude-haiku-4-5-20251001",
            messages=messages,
            timeout=59,
            max_tokens=1024,
            system=system_prompt,
            tools=tools, # List of tools
        )
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason != "tool_use":
            return next(b.text for b in response.content if b.type == "text")

        tool_calls = [block for block in response.content if block.type == "tool_use"]
        results = execute_tool_calls(tool_calls)
        messages.append({"role": "user", "content": results})

In [11]:
todos, completed = [], []
loop(messages)

Todo #1: Determine the distance between Boston and New York
Todo #2: Set up the equation for when the trains meet
Todo #3: Solve for the time when the trains meet
Todo #4: Calculate the exact meeting time

The distance between Boston and New York is approximately 215 miles. This is a well-established distance for this 
major route.

Todo #1: Determine the distance between Boston and New York
Todo #2: Set up the equation for when the trains meet
Todo #3: Solve for the time when the trains meet
Todo #4: Calculate the exact meeting time

Let t = hours after the Boston train departs at 2:00 pm
- Boston train travels 60t miles
- New York train departs at 3:00 pm, so it travels for (t-1) hours
- New York train travels 80(t-1) miles
- They meet when: 60t + 80(t-1) = 215

Todo #1: Determine the distance between Boston and New York
Todo #2: Set up the equation for when the trains meet
Todo #3: Solve for the time when the trains meet
Todo #4: Calculate the exact meeting time

Solving the equation:
60t + 80(t-1) = 215
60t + 80t - 80 = 215
140t = 295
t = 295/140 = 59/28 hours
t ≈ 2.107 hours after 2:00 pm

Todo #1: Determine the distance between Boston and New York
Todo #2: Set up the equation for when the trains meet
Todo #3: Solve for the time when the trains meet
Todo #4: Calculate the exact meeting time

Converting 2.107 hours to hours and minutes:
0.107 hours × 60 minutes/hour ≈ 6.4 minutes
Starting time: 2:00 pm
Meeting time: approximately 4:06 pm (more precisely 4:06:24 pm)

Todo #1: Determine the distance between Boston and New York
Todo #2: Set up the equation for when the trains meet
Todo #3: Solve for the time when the trains meet
Todo #4: Calculate the exact meeting time

'## Solution\n\nThe trains meet at approximately [bold yellow]4:06 PM[/bold yellow].\n\n[bold]Explanation:[/bold]\n- The Boston train departs at 2:00 pm traveling 60 mph\n- The New York train departs at 3:00 pm traveling 80 mph toward Boston\n- Using the distance between Boston and New York of 215 miles\n- When they meet: 60t + 80(t-1) = 215\n- Solving: t ≈ 2.107 hours after 2:00 pm\n- This equals approximately [yellow]4:06 PM[/yellow]'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>